# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [4]:
#!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.14.15

    Uninstalling langsmith-0.14.15:

   ---------- ----------------------------- 1/4 [langsmith]
      Successfully uninstalled langsmith-0.14.15
   ---------- ----------------------------- 1/4 [langsmith]
   ---------- ----------------------------- 1/4 [langsmith]
   ---------- ----------------------------- 1/4 [langsmith]
  Attempting uninstall: langchain-core
   ---------- ----------------------------- 1/4 [langsmith]
    Found existing installation: langchain-core 1.7.1
   ---------- ----------------------------- 1/4 [langsmith]
    Uninstalling langchain-core-1.7.1:
   ---------- ----------------------------- 1/4 [langsmith]
      Successfull

In [3]:
%pip install langchain-elasticsearch

  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ---------------------------------------- 952.8/952.8 kB 4.9 MB/s  0:00:00
   ---------------------------------------- 0.0/575.5 kB ? eta -:--:--
   ---------------------------------------- 575.5/575.5 kB 5.1 MB/s  0:00:00
   ---------------------------------------- 0.0/915.4 kB ? eta -:--:--
   ---------------------------------------- 915.4/915.4 kB 5.2 MB/s  0:00:00
Using cached truststore-0.10.5-py3-none-any.whl (19 kB)

   -------- -------------------------------  3/14 [websockets]
   -------------------- -------------------  7/14 [elastic-transport]
   ---------------------- -----------------  8/14 [httpcore2]
   ------------------------- --------------  9/14 [elasticsearch]
   ------------------------- --------------  9/14 [elasticsearch]
   ------------------------- --------------  9/14 [elasticsearch]
   ------------------------- ---------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.7.1 which is incompatible.
langchain 0.2.17 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.14.15 which is incompatible.
langchain-community 0.2.19 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.7.1 which is incompatible.
langchain-community 0.2.19 requires langsmith<0.2.0,>=0.1.112, but you have langsmith 0.14.15 which is incompatible.
langchain-experimental 0.0.65 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 1.7.1 which is incompatible.
langchain-openai 0.1.25 requires langchain-core<0.3.0,>=0.2.40, but you have langchain-core 1.7.1 which is incompatible.
langchain-pinecone 0.1.3 requires langchain-core<0.3,>=0.1.52, but you have langchain-core 1.7.1 which is incompatib

In [15]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass
from langchain_openai import ChatOpenAI

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [7]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
import os
from dotenv import load_dotenv

# Read the .env file and put its values into environment variables
load_dotenv()

ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-docs", #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [8]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [11]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"]= record.get("summary")
    metadata["url"]= record.get("url")
    metadata["category"]=record.get("category")
    metadata["updated_at"]=record.get("updated_at")
    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [40]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-docs-v2",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [41]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'Who are the members of the NASA sales team and what roles do they play?']


---- Answer ----
content='The NASA sales team refers to the sales organization that operates within the North America and South America region, which is often abbreviated as NASA. This region encompasses the United States, Canada, Mexico, as well as Central and South America. The sales team in this region is structured to effectively serve customers and achieve business objectives specific to these markets.\n\nThe leadership of the NASA sales team includes two Area Vice-Presidents: Laura Martinez, who oversees North America, and Gary Johnson, who is responsible for South America. Each regional sales team, including NASA, consists of dedicated account managers, sales representatives, and support staff. Their primary responsibilities include identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.\n\nThe NASA sales team collaborates closely with other departments, such as marketing, product development, and customer

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [43]:
for d in docs:
    print(d.metadata["name"], "|", d.metadata["category"])

Work From Home Policy | teams
Work From Home Policy | teams
April Work From Home Update | teams
Wfh Policy Update May 2023 | teams
Fy2024 Company Sales Strategy | teams
Fy2024 Company Sales Strategy | teams
Company Vacation Policy | sharepoint
Company Vacation Policy | sharepoint
Swe Career Matrix | sharepoint
Swe Career Matrix | sharepoint
Sales Engineering Collaboration | sharepoint
Sales Engineering Collaboration | sharepoint
Intellectual Property Policy | sharepoint
Intellectual Property Policy | sharepoint
Code Of Conduct | sharepoint
Code Of Conduct | sharepoint
Office Pet Policy | sharepoint
Office Pet Policy | sharepoint
Performance Management Policy | sharepoint
Performance Management Policy | sharepoint
Sales Organization Overview | sharepoint
Compensation Framework For It Teams | sharepoint
Compensation Framework For It Teams | sharepoint
Updating Your Tax Elections Forms | github
Updating Your Tax Elections Forms | github
New Employee Onboarding Guide | github
New Employee 

In [44]:
response = chain.invoke("How many days can I work from home?")
print(response)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the maximum number of days I am allowed to work remotely?  ', '2. How many days per week can I telecommute?  ', '3. What are the guidelines for working from home in my company?']


content="As of May 1, 2023, employees are required to work from the office three days a week, which means they can work from home for two days each week. This adjustment is part of the company's hybrid work policy aimed at balancing in-office collaboration with the flexibility of remote work. If you have any specific questions about your schedule or need to coordinate your remote work days, it's advisable to communicate with your supervisor and the HR department to establish your updated in-office workdays." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 824, 'total_tokens': 921, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2851888338', 'finish_reason': 'stop', 'logprobs': None} id='run-2a593573-ab

In [45]:
docs_found = retriever.invoke("How many days can I work from home?")
for d in docs_found:
    print(d.metadata["name"], "→", d.page_content[:150], "\n")

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the maximum number of days I am allowed to work remotely?  ', '2. How many days per week can I telecommute?  ', '3. What are the guidelines for working from home in my company?']


Wfh Policy Update May 2023 → As we continue to prioritize the well-being of our employees, we are making a slight adjustment to our hybrid work policy. Starting May 1, 2023, emplo 

April Work From Home Update → Starting May 2022, the company will be implementing a two-day in-office work requirement per week for all eligible employees. Please coordinate with y 

Work From Home Policy → Employees must adhere to the company's confidentiality and data security policies while working from home. This includes safeguarding sensitive inform 

Work From Home Policy → Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their 



In [46]:
response2 = chain.invoke("What is the company's vacation policy?")
print(response2.content)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the details of the vacation policy for the company?  ', "Can you provide information about the company's policy on employee vacations?  ", 'What guidelines does the company have regarding vacation time for employees?']


The company's vacation policy is designed to provide guidelines and procedures for employees to request and take time off for personal and leisure purposes, promoting a healthy work-life balance. Here are the key components of the policy:

1. **Purpose**: The primary aim of the vacation policy is to encourage employees to take time off to rest and recharge, thereby enhancing their overall well-being and productivity.

2. **Scope**: This policy applies to all full-time and part-time employees who have completed their probationary period. This means that new employees must first fulfill their initial employment period before they can utilize vacation time.

3. **Vacation Accrual**: 
   - Full-time employees accrue vacation time at a specified rate (denoted as [X hours] per month), which translates to a certain number of days per year (denoted as [Y days]).
   - Part-time employees accrue vacation time on a pro-rata basis, which is calculated based on their scheduled work hours.
   - Vaca

In [47]:
docs_found2 = retriever.invoke("What is the company's vacation policy?")
for d in docs_found2:
    print(d.metadata["name"], "→", d.page_content[:150], "\n")

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the details of the vacation policy for the company?  ', "Can you provide information on the company's policy regarding employee vacations?  ", 'What guidelines does the company have in place for vacation time?']


Company Vacation Policy → This vacation policy will be reviewed periodically and updated as necessary, taking into account changes in labor laws, business needs, and employee f 

Company Vacation Policy → Purpose

The purpose of this vacation policy is to outline the guidelines and procedures for requesting and taking time off from work for personal and 



 ### Bonus:
 Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?

In [48]:
simple_retriever = vectorstore.as_retriever()
print(len(simple_retriever.invoke("How many days can I work from home?")), "chunks (simple)")
print(len(retriever.invoke("How many days can I work from home?")), "chunks (multi-query)")

4 chunks (simple)


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the maximum number of days I am allowed to work remotely?  ', '2. How many days per week can I telecommute?  ', '3. What are the guidelines for working from home in my company?']


4 chunks (multi-query)


In [49]:
retriever = MultiQueryRetriever.from_llm(
    vectorstore.as_retriever(search_kwargs={"k": 2}), llm
)

In [50]:
response2 = chain.invoke("What is the company's vacation policy?")
print(response2.content)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the details of the vacation policy for the company?  ', "Can you provide information about the company's policy on vacation time?  ", 'What guidelines does the company have regarding employee vacations?']


The company's vacation policy is designed to provide guidelines and procedures for employees to request and take time off for personal and leisure purposes, promoting a healthy work-life balance. Here are the key components of the policy:

1. **Purpose**: The policy aims to encourage employees to take time to rest and recharge, thereby enhancing their overall well-being and productivity.

2. **Scope**: This policy applies to all full-time and part-time employees who have completed their probationary period.

3. **Vacation Accrual**: 
   - Full-time employees accrue vacation time at a specified rate (denoted as [X hours] per month), which translates to [Y days] per year.
   - Part-time employees accrue vacation time on a pro-rata basis, based on their scheduled work hours.
   - Vacation time begins to accrue from the first day of employment, but employees can only take vacation after completing their probationary period.
   - Unused vacation time can be carried over to the next year, up

In [34]:
docs_k2 = retriever.invoke("What is the company's vacation policy?")
print(len(docs_k2), "chunks")
for d in docs_k2:
    print(d.metadata["name"], "→", d.page_content[:100], "\n")

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the details of the vacation policy for the company?  ', "Can you provide information on the company's policy regarding employee vacations?  ", 'What guidelines does the company have in place for vacation time?']


1 chunks
Company Vacation Policy → This vacation policy will be reviewed periodically and updated as necessary, taking into account cha 



In [35]:
# What does ONE plain search return with k=2?
plain = vectorstore.as_retriever(search_kwargs={"k": 2}).invoke("What is the company's vacation policy?")
for d in plain:
    print(d.metadata["name"], "→", d.page_content[:80])

# How many chunks are stored in the index in total?
print(vectorstore.client.count(index="workplace-docs")["count"])

Company Vacation Policy → This vacation policy will be reviewed periodically and updated as necessary, tak
Company Vacation Policy → This vacation policy will be reviewed periodically and updated as necessary, tak
54


In [28]:
from langchain_core.prompts import ChatPromptTemplate

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """Use the following context to answer the question at the end.
If the answer is not in the context, just say "I don't know". Don't make up an answer.

Answer format:
- Maximum 3 short bullet points
- Each bullet under 20 words
- Friendly and simple language

----
{context}
----
Question: {question}
Answer:"""
)

In [29]:
print(chain.invoke("How many days can I work from home?"))

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the maximum number of days I am allowed to work remotely?  ', '2. How many days per week can I telecommute?  ', '3. What are the guidelines for working from home in my company?']


content="As of May 1, 2023, employees are required to work from the office three days a week, which means they can work from home for two days each week. This adjustment is part of the company's hybrid work policy aimed at balancing in-office collaboration with the flexibility of remote work. If you have any specific questions about your schedule or need to coordinate your remote work days, it's advisable to communicate with your supervisor and the HR department to establish your updated in-office workdays." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 824, 'total_tokens': 921, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2851888338', 'finish_reason': 'stop', 'logprobs': None} id='run-a4848098-ac

## Reflection

**Question 1: "How many days can I work from home?"**
The generated queries were useful: they used different words ("remotely", "telecommute").
The answer (2 days from home since May 2023) was correct. The retriever also found older,
conflicting policies, but the model chose the newest one.

**Question 2: "What is the company's vacation policy?"**
The generated queries were very similar to each other, so multi-query added little benefit.
The answer was grounded but not very useful: the source document is a template with
placeholders like [X days], so it couldn't say how many vacation days employees get.
Two of the four retrieved chunks were irrelevant (work from home), but the model ignored them.

**Bonus B: changing k**
With k=2 I got only 1 unique chunk, because my index had duplicates (54 instead of 27)
from running the import cell twice. After fixing the index, k=2 returned [N] chunks and
the answer [was shorter / stayed the same].


**Lessons learned**
- MultiQueryRetriever helps most when the generated queries use really different words.
- A RAG answer is only as good as its documents (outdated or incomplete data gives weak answers).
- Duplicate data in the vector database wastes the k slots and reduces retrieval quality.